# MiDaS Depth Estimation — TUM Dataset

## Estrutura de saída
```
orbslam_midas/<dataset>/
├── raw/                  ← float32 bruto (.npy) — dados originais do MiDaS
├── depth_midas/          ← uint16 PNG normalizado → pronto para ORB-SLAM3
├── associations_midas.txt
└── stats_<dataset>.json
```

## Pipeline completa
1. **Inferência** → salva `.npy` bruto
2. **Pós-processamento** → normaliza e converte para uint16 TUM


In [ ]:
# Verificar GPU e montar Drive
import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')
print('Drive montado!')

GPU: True
Device: Tesla T4
VRAM: 14.6 GB
Mounted at /content/drive
Drive montado!


In [ ]:
#  Instalar dependências
!pip install timm -q

import os, sys, time, json, shutil
import numpy as np
from PIL import Image
import cv2
import torch

print(f'PyTorch  : {torch.__version__}')
print(f'NumPy    : {np.__version__}')
print('Dependências OK!')

PyTorch  : 2.11.0+cu128
NumPy    : 2.0.2
Dependências OK!


In [ ]:
# CONFIGURAÇÕES 

# Dataset — comece com fr2_xyz, depois troque para fr1_desk
DATASET = 'fr3_office'   # 'fr2_xyz' | 'fr1_desk' | 'fr3_office'

# Modelo MiDaS
MIDAS_MODEL  = 'DPT_Large'  # melhor qualidade
MODEL_TYPE   = 'midas'

# Fator de codificação para uint16 (convenção TUM: 1m = 5000 unidades)
DEPTH_FACTOR = 5000.0

DATASET_CONFIG = {
    'fr2_xyz': {
        'url':    'https://cvg.cit.tum.de/rgbd/dataset/freiburg2/rgbd_dataset_freiburg2_xyz.tgz',
        'folder': 'rgbd_dataset_freiburg2_xyz',
    },
    'fr1_desk': {
        'url':    'https://cvg.cit.tum.de/rgbd/dataset/freiburg1/rgbd_dataset_freiburg1_desk.tgz',
        'folder': 'rgbd_dataset_freiburg1_desk',
    },
    'fr3_office': {
        'url':    'https://cvg.cit.tum.de/rgbd/dataset/freiburg3/rgbd_dataset_freiburg3_long_office_household.tgz',
        'folder': 'rgbd_dataset_freiburg3_long_office_household',
    },
}

cfg          = DATASET_CONFIG[DATASET]
DATASET_DIR  = f'/content/{cfg["folder"]}'

# Pastas de saída no Drive
DRIVE_BASE   = f'/content/drive/MyDrive/orbslam_midas/{DATASET}'
DRIVE_RAW    = f'{DRIVE_BASE}/raw'          # float32 .npy bruto
DRIVE_PROC   = f'{DRIVE_BASE}/processed'    # ZIPs processados
for p in [DRIVE_BASE, DRIVE_RAW, DRIVE_PROC]:
    os.makedirs(p, exist_ok=True)

print(f'Dataset      : {DATASET}')
print(f'Modelo       : {MIDAS_MODEL}')
print(f'DEPTH_FACTOR : {DEPTH_FACTOR}')
print(f'Drive raw    : {DRIVE_RAW}')
print(f'Drive proc   : {DRIVE_PROC}')

Dataset      : fr3_office
Modelo       : DPT_Large
DEPTH_FACTOR : 5000.0
Drive raw    : /content/drive/MyDrive/orbslam_midas/fr3_office/raw
Drive proc   : /content/drive/MyDrive/orbslam_midas/fr3_office/processed


In [ ]:
# Baixar e extrair dataset TUM
if not os.path.exists(DATASET_DIR):
    print(f'Baixando {DATASET}...')
    !wget -q --show-progress {cfg['url']} -O /content/dataset.tgz
    print('Extraindo...')
    !tar -xzf /content/dataset.tgz -C /content/
    !rm /content/dataset.tgz
    print('Dataset pronto!')
else:
    print(f'Dataset já existe: {DATASET_DIR}')

print(f'RGB   : {len(os.listdir(f"{DATASET_DIR}/rgb"))} imagens')
print(f'Depth : {len(os.listdir(f"{DATASET_DIR}/depth"))} imagens')

Baixando fr3_office...
/content/dataset.tg 100%[===================>]   1.38G  20.1MB/s    in 72s     
Extraindo...
Dataset pronto!
RGB   : 2585 imagens
Depth : 2509 imagens


In [ ]:
# Carregar modelo MiDaS
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

print(f'Carregando MiDaS {MIDAS_MODEL}...')
midas     = torch.hub.load('intel-isl/MiDaS', MIDAS_MODEL)
midas.eval().to(DEVICE)

transforms = torch.hub.load('intel-isl/MiDaS', 'transforms')
transform  = transforms.dpt_transform  # DPT_Large / DPT_Hybrid

print(f'MiDaS {MIDAS_MODEL} carregado!')
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM: {used:.2f} GB / {total:.1f} GB')

Device: cuda
Carregando MiDaS DPT_Large...
The repository intel-isl_MiDaS does not belong to the list of trusted repositories and as such cannot be downloaded. Do you trust this repository and wish to add it to the trusted list of repositories (y/N)?y
Downloading: "https://github.com/intel-isl/MiDaS/zipball/master" to /root/.cache/torch/hub/master.zip


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Downloading: "https://github.com/isl-org/MiDaS/releases/download/v3/dpt_large_384.pt" to /root/.cache/torch/hub/checkpoints/dpt_large_384.pt


100%|██████████| 1.28G/1.28G [00:09<00:00, 138MB/s]


MiDaS DPT_Large carregado!
VRAM: 1.28 GB / 14.6 GB


Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


In [ ]:
#  Inferência.npy BRUTO (float32)
# Sem normalização, sem conversão — dados originais do MiDaS
from tqdm.notebook import tqdm

# Ler lista de imagens
rgb_txt     = os.path.join(DATASET_DIR, 'rgb.txt')
rgb_entries = []
with open(rgb_txt) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        parts = line.split()
        if len(parts) >= 2:
            rgb_entries.append((parts[0], parts[1]))

print(f'Total de frames: {len(rgb_entries)}')

# Pasta local para .npy
RAW_LOCAL = os.path.join(DATASET_DIR, 'depth_midas_raw')
os.makedirs(RAW_LOCAL, exist_ok=True)

ts_list = []   # para usar na célula de pós-processamento
times   = []
raw_stats = []  # stats dos primeiros frames para inspeção

for timestamp, rel_path in tqdm(rgb_entries, desc='MiDaS inferência'):
    rgb_path = os.path.join(DATASET_DIR, rel_path)
    if not os.path.exists(rgb_path):
        continue

    t0 = time.time()

    img     = cv2.imread(rgb_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    orig_h, orig_w = img_rgb.shape[:2]

    input_batch = transform(img_rgb).to(DEVICE)
    with torch.no_grad():
        prediction = midas(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=(orig_h, orig_w),
            mode='bicubic',
            align_corners=False
        ).squeeze()

    # Salvar float32 bruto
    disp = prediction.cpu().numpy().astype(np.float32)
    np.save(os.path.join(RAW_LOCAL, f'{timestamp}.npy'), disp)

    ts_list.append((timestamp, rel_path, orig_w, orig_h))
    times.append(time.time() - t0)

    if len(raw_stats) < 5:
        raw_stats.append({
            'ts':   timestamp,
            'min':  float(disp.min()),
            'max':  float(disp.max()),
            'mean': float(disp.mean()),
            'std':  float(disp.std()),
        })

avg_ms = np.mean(times) * 1000
print(f'\nInferência concluída!')
print(f'Frames salvos (.npy) : {len(ts_list)}')
print(f'Latência média       : {avg_ms:.1f} ms/frame ({1000/avg_ms:.1f} FPS)')
print(f'\nStats dos primeiros frames (disparidade bruta float32):')
for s in raw_stats:
    print(f"  {s['ts']}: min={s['min']:.2f} max={s['max']:.2f} "
          f"mean={s['mean']:.2f} std={s['std']:.2f}")

Total de frames: 2585


MiDaS inferência:   0%|          | 0/2585 [00:00<?, ?it/s]


Inferência concluída!
Frames salvos (.npy) : 2585
Latência média       : 263.5 ms/frame (3.8 FPS)

Stats dos primeiros frames (disparidade bruta float32):
  1341847980.722988: min=-0.24 max=35.84 mean=16.65 std=6.50
  1341847980.754743: min=-0.31 max=35.46 mean=16.66 std=6.52
  1341847980.786856: min=-0.37 max=35.39 mean=16.69 std=6.49
  1341847980.822978: min=-0.32 max=35.94 mean=16.64 std=6.58
  1341847980.854676: min=-0.13 max=35.62 mean=16.54 std=6.58


In [ ]:
# CÉLULA 7 — Pós-processamento: .npy → uint16 PNG (formato TUM)
# Usa range completo uint16 [0, 65535] 

DEPTH_OUT    = os.path.join(DATASET_DIR, 'depth_midas')
depth_folder = 'depth_midas'
os.makedirs(DEPTH_OUT, exist_ok=True)

# Fator para recuperar a disparidade normalizada no ORB-SLAM3
# uint16 / 65535 = disparidade [0, 1]
UINT16_MAX = 65535

associations = []

print('Convertendo .npy → uint16 PNG (range completo)...')
for timestamp, rel_path, orig_w, orig_h in tqdm(ts_list, desc='Pós-processamento'):
    npy_path = os.path.join(RAW_LOCAL, f'{timestamp}.npy')
    if not os.path.exists(npy_path):
        continue

    disp = np.load(npy_path)  # float32 bruto

    # Normalizar para [0, 1] por frame
    d_min, d_max = disp.min(), disp.max()
    if d_max - d_min > 1e-6:
        disp_norm = (disp - d_min) / (d_max - d_min)
    else:
        disp_norm = np.zeros_like(disp)

    # Usar range completo uint16 — melhor precisão
    depth_uint16 = (disp_norm * UINT16_MAX).astype(np.uint16)

    depth_filename = f'{timestamp}.png'
    Image.fromarray(depth_uint16).save(os.path.join(DEPTH_OUT, depth_filename))

    associations.append(
        f'{timestamp} rgb/{os.path.basename(rel_path)} '
        f'{timestamp} depth_midas/{depth_filename}'
    )

print(f'Frames convertidos: {len(associations)}')

print('\n=== DEPTH MIDAS uint16 range completo ===')
for fname in sorted(os.listdir(DEPTH_OUT))[:3]:
    img = np.array(Image.open(os.path.join(DEPTH_OUT, fname)))
    print(f'  {fname}: min={img.min()} max={img.max()} mean={img.mean():.0f} -> {img.mean()/5000:.2f}m dtype={img.dtype}')

print('\n=== DEPTH REAL TUM ===')
depth_real_dir = os.path.join(DATASET_DIR, 'depth')
for fname in sorted(os.listdir(depth_real_dir))[:3]:
    img = np.array(Image.open(os.path.join(depth_real_dir, fname)))
    print(f'  {fname}: min={img.min()} max={img.max()} mean={img.mean():.0f} '
          f'→ {img.mean()/5000:.2f}m dtype={img.dtype}')

print(f'\nNOTA: usar DepthMapFactor={UINT16_MAX} no YAML do ORB-SLAM3')

Convertendo .npy → uint16 PNG (range completo)...


Pós-processamento:   0%|          | 0/2585 [00:00<?, ?it/s]

Frames convertidos: 2585

=== DEPTH MIDAS uint16 range completo ===
  1341847980.722988.png: min=0 max=65535 mean=30680 -> 6.14m dtype=uint16
  1341847980.754743.png: min=0 max=65535 mean=31096 -> 6.22m dtype=uint16
  1341847980.786856.png: min=0 max=65535 mean=31262 -> 6.25m dtype=uint16

=== DEPTH REAL TUM ===
  1341847980.723020.png: min=0 max=46655 mean=10009 → 2.00m dtype=uint16
  1341847980.754755.png: min=0 max=49350 mean=10423 → 2.08m dtype=uint16
  1341847980.786879.png: min=0 max=49350 mean=10361 → 2.07m dtype=uint16

NOTA: usar DepthMapFactor=65535 no YAML do ORB-SLAM3


In [ ]:
# Salvar tudo no Drive

# 1. Associations
assoc_filename = 'associations_midas.txt'
assoc_local    = os.path.join(DATASET_DIR, assoc_filename)
with open(assoc_local, 'w') as f:
    f.write('\n'.join(associations))
shutil.copy(assoc_local, os.path.join(DRIVE_PROC, assoc_filename))
print(f'Associations: {len(associations)} pares')

# 2. Stats JSON
stats = {
    'modelo':            MODEL_TYPE,
    'midas_variant':     MIDAS_MODEL,
    'dataset':           DATASET,
    'depth_factor':      DEPTH_FACTOR,
    'frames':            len(associations),
    'latencia_media_ms': float(np.mean(times) * 1000),
    'fps_medio':         float(1000 / (np.mean(times) * 1000)),
    'raw_stats_amostra': raw_stats,
    'nota':              'raw=float32 bruto; depth_midas=uint16 normalizado por frame; calibracao affine no host'
}
with open(os.path.join(DRIVE_PROC, f'stats_{DATASET}_midas.json'), 'w') as f:
    json.dump(stats, f, indent=2)
print(f'Stats salvo!')

# 3. ZIP depth_midas (uint16 processado)
print('\nCompactando depth_midas (uint16)...')
zip_proc = os.path.join(DRIVE_PROC, f'depth_midas_{DATASET}')
shutil.make_archive(zip_proc, 'zip', DATASET_DIR, 'depth_midas')
print(f'ZIP processado: {zip_proc}.zip')

# 4. ZIP depth_midas_raw (.npy bruto)
print('Compactando depth_midas_raw (.npy)...')
zip_raw = os.path.join(DRIVE_RAW, f'depth_midas_raw_{DATASET}')
shutil.make_archive(zip_raw, 'zip', DATASET_DIR, 'depth_midas_raw')
print(f'ZIP bruto: {zip_raw}.zip')

print('\n=== DRIVE — processado ===')
!ls -lh {DRIVE_PROC}
print('\n=== DRIVE — raw ===')
!ls -lh {DRIVE_RAW}

Associations: 2585 pares
Stats salvo!

Compactando depth_midas (uint16)...
ZIP processado: /content/drive/MyDrive/orbslam_midas/fr3_office/processed/depth_midas_fr3_office.zip
Compactando depth_midas_raw (.npy)...
ZIP bruto: /content/drive/MyDrive/orbslam_midas/fr3_office/raw/depth_midas_raw_fr3_office.zip

=== DRIVE — processado ===
total 833M
-rw------- 1 root root 243K Jun 27 13:24 associations_midas.txt
-rw------- 1 root root 832M Jun 27 13:25 depth_midas_fr3_office.zip
-rw------- 1 root root 646K Jun 27 13:24 preview_midas_fr3_office.pdf
-rw------- 1 root root 1.2K Jun 27 13:24 stats_fr3_office_midas.json

=== DRIVE — raw ===
total 2.5G
-rw------- 1 root root 2.5G Jun 27 13:29 depth_midas_raw_fr3_office.zip
